In [5]:
import numpy as np
import pandas as pd
import re

df = pd.read_csv(r'C:\Users\A.R.I\Desktop\Data_Analysis\matchs_grok.csv')

def safe_text(x):
    if isinstance(x, str):
        return x
    return ""

emoji_pattern = re.compile(
    "["                              
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # flags
    "]+", flags=re.UNICODE
)

def count_emojis(text):
    text = safe_text(text)
    return len(emoji_pattern.findall(text))

def count_code_blocks(text):
    text = safe_text(text)
    return text.count("```")

def count_sentences(text):
    text = safe_text(text)
    # جمله‌های خیلی ساده با . ! ? 
    return len(re.findall(r"[.!?]+", text))

def count_punct(text):
    text = safe_text(text)
    return sum(1 for c in text if c in ".,!?;:")


In [6]:
def prepare_base(df_raw: pd.DataFrame, win_rate_threshold: float = 35.0):
    """
    df_raw : دیتافریم اصلی که از CSV لود شده (همون df)
    win_rate_threshold : آستانه‌ی weak-opponent (پیش‌فرض ۳۵٪)
    """
    
    df = df_raw[(df_raw["grok_in_a"]) | (df_raw["grok_in_b"])].copy()

    
    df["opponent"] = np.where(df["grok_in_a"], df["model_b"], df["model_a"])

    
    
    df["is_grok_win"] = (
        ((df["winner"] == "model_a") & df["grok_in_a"]) |
        ((df["winner"] == "model_b") & df["grok_in_b"])
    )

   
    df["is_grok_tie"] = (df["winner"] == "tie")

    
    df["is_both_bad"] = (df["winner"] == "both_bad")

  
    df["is_grok_loss"] = (
        (~df["is_grok_win"] & ~df["is_grok_tie"]) | df["is_both_bad"]
    )

    
    rivals = (
        df.groupby("opponent")
          .agg(
              matches=("id", "count"),
              wins=("is_grok_win", "sum"),
              ties=("is_grok_tie", "sum"),
              losses=("is_grok_loss", "sum")
          )
          .reset_index()
    )
    rivals["win_rate(%)"] = (rivals["wins"] / rivals["matches"] * 100).round(1)

    
    weak_opponents = rivals[rivals["win_rate(%)"] < win_rate_threshold].copy()

   
    weak_losses = df[
        (df["opponent"].isin(weak_opponents["opponent"])) &
        (df["is_grok_loss"])
    ].copy()

    return df, rivals, weak_opponents, weak_losses


In [7]:
def extract_features(weak_losses: pd.DataFrame) -> pd.DataFrame:
    """
    برای هر باخت Grok در برابر weak_opponent، فیچرهای Grok و حریف را استخراج می‌کند.
    """
    rows = []

    for _, row in weak_losses.iterrows():
        
        if row["grok_in_a"]:
            grok_text = safe_text(row.get("response_a"))
            opp_text  = safe_text(row.get("response_b"))
        else:
            grok_text = safe_text(row.get("response_b"))
            opp_text  = safe_text(row.get("response_a"))

        full_conv = safe_text(row.get("full_conversation"))
        meta = row.get("conv_metadata")
        meta_str = safe_text(str(meta))

        rows.append({
            
            "opponent": row.get("opponent"),
            "language": row.get("language"),
            "category": row.get("category_tag"),
            "is_code_prompt": row.get("is_code"),

            
            "grok_len_chars": len(grok_text),
            "grok_len_words": len(grok_text.split()),
            "grok_sentences": count_sentences(grok_text),
            "grok_emojis": count_emojis(grok_text),
            "grok_code_blocks": count_code_blocks(grok_text),
            "grok_punct": count_punct(grok_text),

           
            "opp_len_chars": len(opp_text),
            "opp_len_words": len(opp_text.split()),
            "opp_sentences": count_sentences(opp_text),
            "opp_emojis": count_emojis(opp_text),
            "opp_code_blocks": count_code_blocks(opp_text),
            "opp_punct": count_punct(opp_text),

            
            "len_chars_ratio_grok_opp": (
                len(grok_text) / max(1, len(opp_text))
            ),
            "len_words_ratio_grok_opp": (
                len(grok_text.split()) / max(1, len(opp_text.split()))
            ),

            
            "conversation_length_chars": len(full_conv),
            "conversation_turns": row.get("turns", 0),

           
            "meta_length": len(meta_str),
            "meta_has_emoji": count_emojis(meta_str) > 0,
        })

    return pd.DataFrame(rows)


In [8]:
df_grok_all, rivals, weak_opponents, weak_losses = prepare_base(df, win_rate_threshold=35.0)

print("Total Grok matches:", len(df_grok_all))
print("Total weak opponents:", len(weak_opponents))
print("Total Grok losses vs weak opponents:", len(weak_losses))


df_features = extract_features(weak_losses)

df_features.head(10)


Total Grok matches: 5180
Total weak opponents: 6
Total Grok losses vs weak opponents: 358


,opponent,language,category,is_code_prompt,grok_len_chars,grok_len_words,grok_sentences,grok_emojis,grok_code_blocks,grok_punct,...,opp_sentences,opp_emojis,opp_code_blocks,opp_punct,len_chars_ratio_grok_opp,len_words_ratio_grok_opp,conversation_length_chars,conversation_turns,meta_length,meta_has_emoji
0,o3-2025-04-16,en,{'creative_writing_v0.1': {'creative_writing':...,True,765,121,10,0,2,16,...,1,0,2,7,2.637931,2.750000,2505,1,460,False
1,gemini-2.5-pro-preview-05-06,en,{'creative_writing_v0.1': {'creative_writing':...,True,9073,1094,88,0,22,154,...,154,0,18,265,0.681003,0.628014,37464,1,468,False
2,gemini-2.5-pro-preview-05-06,en,{'creative_writing_v0.1': {'creative_writing':...,False,8598,1262,156,0,0,321,...,58,0,0,135,1.848635,1.984277,22825,1,467,False
3,gemini-2.5-pro-preview-03-25,en,{'creative_writing_v0.1': {'creative_writing':...,True,11562,1044,146,0,2,468,...,235,12,2,750,0.536147,0.577114,57185,1,465,False
4,gemini-2.5-pro-preview-05-06,en,{'creative_writing_v0.1': {'creative_writing':...,True,7007,1017,91,0,0,177,...,67,0,0,145,1.248352,1.300512,20468,1,466,False
5,o3-2025-04-16,und,{'creative_writing_v0.1': {'creative_writing':...,False,339,60,3,0,0,7,...,2,1,0,2,6.780000,6.000000,1273,1,456,False
6,gemini-2.5-pro,fr,{'creative_writing_v0.1': {'creative_writing':...,False,2193,396,18,0,0,48,...,10,0,0,13,4.291585,4.658824,5757,1,461,False
7,chatgpt-4o-latest-20250326,en,{'creative_writing_v0.1': {'creative_writing':...,False,78,11,1,0,0,4,...,2,0,0,3,0.795918,0.785714,844,1,456,False
8,gemini-2.5-pro,cs,{'creative_writing_v0.1': {'creative_writing':...,False,2709,380,27,0,0,58,...,39,0,0,81,0.735142,0.695971,10839,1,465,False
9,deepseek-r1-0528,und,{'creative_writing_v0.1': {'creative_writing':...,True,3259,352,29,0,16,50,...,25,1,14,33,1.593643,1.289377,9543,1,465,False


In [9]:
df_features.describe().T

,count,mean,std,min,25%,50%,75%,max
grok_len_chars,358.0,4250.120112,3355.933379,0.0,1483.250000,3644.000000,6454.250000,18504.0
grok_len_words,358.0,598.086592,480.635569,0.0,190.250000,515.500000,936.500000,2479.0
grok_sentences,358.0,47.547486,44.635363,0.0,11.250000,35.000000,73.000000,217.0
grok_emojis,358.0,0.212291,0.805906,0.0,0.000000,0.000000,0.000000,9.0
grok_code_blocks,358.0,1.379888,4.086712,0.0,0.000000,0.000000,0.000000,32.0
grok_punct,358.0,114.480447,105.908308,0.0,29.250000,88.500000,167.750000,626.0
opp_len_chars,358.0,3475.896648,3208.997650,2.0,1383.000000,2889.000000,4756.750000,32540.0
opp_len_words,358.0,470.818436,373.540241,1.0,183.250000,401.500000,679.500000,2564.0
opp_sentences,358.0,41.477654,39.735728,0.0,14.000000,31.000000,57.500000,253.0
opp_emojis,358.0,0.921788,2.268904,0.0,0.000000,0.000000,1.000000,14.0


In [11]:
lang_counts = weak_losses['language'].value_counts().sort_values(ascending=False)

print("Grok losses in weak-opponent battles (per language):")
lang_counts

Grok losses in weak-opponent battles (per language):


language
en         194
und         30
pl          26
ru          22
zh          18
de          13
ja           9
ko           8
fr           7
es           5
tr           4
cs           3
ca           2
sv           2
it           2
la           1
hu           1
pt           1
vi           1
ms           1
ro           1
ne           1
kk           1
zh-Hant      1
nl           1
fa           1
uk           1
id           1
Name: count, dtype: int64

In [12]:
valid_langs = lang_counts[lang_counts >= 10].index
valid_langs

Index(['en', 'und', 'pl', 'ru', 'zh', 'de'], dtype='object', name='language')

In [13]:
df_features_filtered = df_features[df_features["language"].isin(valid_langs)]
df_features_filtered.head()

,opponent,language,category,is_code_prompt,grok_len_chars,grok_len_words,grok_sentences,grok_emojis,grok_code_blocks,grok_punct,...,opp_sentences,opp_emojis,opp_code_blocks,opp_punct,len_chars_ratio_grok_opp,len_words_ratio_grok_opp,conversation_length_chars,conversation_turns,meta_length,meta_has_emoji
0,o3-2025-04-16,en,{'creative_writing_v0.1': {'creative_writing':...,True,765,121,10,0,2,16,...,1,0,2,7,2.637931,2.750000,2505,1,460,False
1,gemini-2.5-pro-preview-05-06,en,{'creative_writing_v0.1': {'creative_writing':...,True,9073,1094,88,0,22,154,...,154,0,18,265,0.681003,0.628014,37464,1,468,False
2,gemini-2.5-pro-preview-05-06,en,{'creative_writing_v0.1': {'creative_writing':...,False,8598,1262,156,0,0,321,...,58,0,0,135,1.848635,1.984277,22825,1,467,False
3,gemini-2.5-pro-preview-03-25,en,{'creative_writing_v0.1': {'creative_writing':...,True,11562,1044,146,0,2,468,...,235,12,2,750,0.536147,0.577114,57185,1,465,False
4,gemini-2.5-pro-preview-05-06,en,{'creative_writing_v0.1': {'creative_writing':...,True,7007,1017,91,0,0,177,...,67,0,0,145,1.248352,1.300512,20468,1,466,False


In [16]:
lang_summary = (
    df_features_filtered
    .groupby("language")
    .agg({
        # --- Response length ---
        "grok_len_chars": "mean",
        "opp_len_chars": "mean",
        "grok_len_words": "mean",
        "opp_len_words": "mean",
        "grok_sentences": "mean",
        "opp_sentences": "mean",

        # --- Structure features ---
        "grok_emojis": "mean",
        "opp_emojis": "mean",
        "grok_code_blocks": "mean",
        "opp_code_blocks": "mean",
        "grok_punct": "mean",
        "opp_punct": "mean",

        # --- Ratios ---
        "len_chars_ratio_grok_opp": "mean",
        "len_words_ratio_grok_opp": "mean",

        # --- Conversation-level ---
        "conversation_length_chars": "mean",
        "conversation_turns": "mean",

        # --- Metadata ---
        "meta_length": "mean",
        "meta_has_emoji": "mean",
    })
    .round(2)
)

lang_summary


,grok_len_chars,opp_len_chars,grok_len_words,opp_len_words,grok_sentences,opp_sentences,grok_emojis,opp_emojis,grok_code_blocks,opp_code_blocks,grok_punct,opp_punct,len_chars_ratio_grok_opp,len_words_ratio_grok_opp,conversation_length_chars,conversation_turns,meta_length,meta_has_emoji
language,,,,,,,,,,,,,,,,,,
de,3423.38,3415.85,457.62,455.38,39.54,37.23,0.15,0.00,0.00,0.00,86.31,79.31,1.01,1.00,11502.23,1.0,464.00,0.0
en,4830.74,3643.50,714.85,515.45,56.20,46.31,0.16,0.84,1.59,1.78,137.02,103.52,1.87,1.86,15108.40,1.0,463.39,0.0
pl,3975.88,3408.69,525.31,433.65,48.42,41.35,0.65,2.00,1.38,1.15,113.27,98.35,1.59,1.57,12939.15,1.0,464.46,0.0
ru,4229.18,3548.86,578.50,481.09,45.77,43.14,0.45,0.95,0.64,1.55,115.36,100.00,1.28,1.28,13584.50,1.0,465.32,0.0
und,2774.67,2856.00,340.93,349.73,26.93,29.63,0.17,0.87,2.13,1.73,65.77,73.50,1.26,1.20,10024.77,1.0,463.13,0.0
zh,2734.61,3869.00,234.00,322.11,19.61,25.50,0.06,0.39,0.78,0.89,32.11,58.33,1.23,1.23,11807.00,1.0,466.11,0.0


In [17]:
lang_counts = weak_losses['language'].value_counts()
lang_counts

language
en         194
und         30
pl          26
ru          22
zh          18
de          13
ja           9
ko           8
fr           7
es           5
tr           4
cs           3
ca           2
sv           2
it           2
la           1
hu           1
pt           1
vi           1
ms           1
ro           1
ne           1
kk           1
zh-Hant      1
nl           1
fa           1
uk           1
id           1
Name: count, dtype: int64

In [18]:
lang_summary

,grok_len_chars,opp_len_chars,grok_len_words,opp_len_words,grok_sentences,opp_sentences,grok_emojis,opp_emojis,grok_code_blocks,opp_code_blocks,grok_punct,opp_punct,len_chars_ratio_grok_opp,len_words_ratio_grok_opp,conversation_length_chars,conversation_turns,meta_length,meta_has_emoji
language,,,,,,,,,,,,,,,,,,
de,3423.38,3415.85,457.62,455.38,39.54,37.23,0.15,0.00,0.00,0.00,86.31,79.31,1.01,1.00,11502.23,1.0,464.00,0.0
en,4830.74,3643.50,714.85,515.45,56.20,46.31,0.16,0.84,1.59,1.78,137.02,103.52,1.87,1.86,15108.40,1.0,463.39,0.0
pl,3975.88,3408.69,525.31,433.65,48.42,41.35,0.65,2.00,1.38,1.15,113.27,98.35,1.59,1.57,12939.15,1.0,464.46,0.0
ru,4229.18,3548.86,578.50,481.09,45.77,43.14,0.45,0.95,0.64,1.55,115.36,100.00,1.28,1.28,13584.50,1.0,465.32,0.0
und,2774.67,2856.00,340.93,349.73,26.93,29.63,0.17,0.87,2.13,1.73,65.77,73.50,1.26,1.20,10024.77,1.0,463.13,0.0
zh,2734.61,3869.00,234.00,322.11,19.61,25.50,0.06,0.39,0.78,0.89,32.11,58.33,1.23,1.23,11807.00,1.0,466.11,0.0


In [19]:
df_features.describe().T

,count,mean,std,min,25%,50%,75%,max
grok_len_chars,358.0,4250.120112,3355.933379,0.0,1483.250000,3644.000000,6454.250000,18504.0
grok_len_words,358.0,598.086592,480.635569,0.0,190.250000,515.500000,936.500000,2479.0
grok_sentences,358.0,47.547486,44.635363,0.0,11.250000,35.000000,73.000000,217.0
grok_emojis,358.0,0.212291,0.805906,0.0,0.000000,0.000000,0.000000,9.0
grok_code_blocks,358.0,1.379888,4.086712,0.0,0.000000,0.000000,0.000000,32.0
grok_punct,358.0,114.480447,105.908308,0.0,29.250000,88.500000,167.750000,626.0
opp_len_chars,358.0,3475.896648,3208.997650,2.0,1383.000000,2889.000000,4756.750000,32540.0
opp_len_words,358.0,470.818436,373.540241,1.0,183.250000,401.500000,679.500000,2564.0
opp_sentences,358.0,41.477654,39.735728,0.0,14.000000,31.000000,57.500000,253.0
opp_emojis,358.0,0.921788,2.268904,0.0,0.000000,0.000000,1.000000,14.0
